# Project: Intelligent Retail (Edge Model)
### Datum: 2026-03-15
### Team: Michal Kakol

## 0. Setup:

We use PySpark for the Cloud pipeline to handle high Volume and Velocity (Big Data 3 V's). For the Edge, we use PIL and Pandas to keep the footprint small for local hardware.

In [ ]:
import logging
import os
import pandas as pd
from PIL import Image
import pickle
import hashlib
import json
from datetime import datetime
logging.basicConfig(level=logging.DEBUG)

### 0.1. Edge:

In [ ]:
from functions.edge import *

### 0.2. Cloud:

In [ ]:
from functions.cloud import *

## 1. Data Loading:

We separate our ingestion into two streams:

    Edge: Focused on Variety (unstructured image data).

    Cloud: Focused on Volume (structured historical sales).
    We use a Schema (Blueprint) for Spark to ensure data quality and faster loading.

### 1.1. Edge:

In [ ]:
processed_images, labels = edge_load_process("../data/raw/image/frames", "../data/raw/image/labels.csv")

### 1.2. Cloud:

We use Spark SQL concepts here. The loading process is Lazy, meaning Spark only plans the execution until an Action (like count) is called.

In [ ]:
df_cloud = cloud_load_transform(spark, "../data/raw/text/train.csv")

## 2. Data Processing:

monitoring the pipeline to detect changes in data during processing

### 2.1. Edge:

To ensure the model is "lightweight," images are resized locally. This is a form of Vertical Scaling (optimizing resources on a single node).

In [ ]:
edge_stats = edge_save_monitor(processed_images, labels, "../data/processed/images.pkl", "../data/monitoring/edge_stats.json")

### 2.2. Cloud:

We use .coalesce(1) before saving to Parquet. This avoids Shuffling (moving data between cluster nodes) and prevents the small file problem, which slows down downstream ML training.

In [ ]:
cloud_stats = cloud_save_monitor(df_cloud, "../data/processed/sales.parquet", "../data/monitoring/cloud_stats.json")

## 3. Pipeline:

These functions encapsulate the entire logic for a production-ready "Continuous Integration" (CI) setup.

### 3.1. Edge:

Triggered by Event-Based logic when new camera frames are available.

In [ ]:
edge_pipeline("../data/raw/image/frames", "../data/raw/image/labels.csv", "../data/processed/images.pkl")

### 3.1. Cloud:

Handles Batch Processing of sales data. It uses the defined StructType schema to validate data integrity automatically.

In [ ]:
cloud_pipeline(spark, "../data/raw/text/train.csv", "../data/processed/sales.parquet")